<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Gymnasiet_Lab_2_Maskininlarning/Lektion_4_Utvardering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Maskininlärning – Lektion 4: Hur bra blev AI:n?

**Målgrupp:** Gymnasiet, 16 år, inga förkunskaper krävs  
**Tid:** ca 45 minuter  
**Mål:** Förstå hur man utvärderar en AI-modell med noggrannhet och förväxlingsmatris

---

### Upphovspersoner
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se  
Gymnasieversion baserad på originalverket ovan.

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/

---
## 📝 Del 1 – Att rätta provet

I Lektion 2 sparade vi 30 blommor som **testdata (Test Data)** –  
det "hemliga provet" som AI:n inte fick se under träningen.

Nu är det dags att låta AI:n göra provet och sedan **rätta** det!

```
PROVET                          RÄTTNING
──────────────────────────      ──────────────────────────
Blomma 1: [5.8, 4.0, 1.2, 0.2] → AI gissade: Setosa  | Rätt svar: Setosa  ✓
Blomma 2: [5.7, 2.6, 3.5, 1.0] → AI gissade: Setosa  | Rätt svar: Versicolor ✗
Blomma 3: [6.7, 3.0, 5.2, 2.3] → AI gissade: Virginica| Rätt svar: Virginica ✓
...
```

Vi kan sedan räkna: av 30 frågor, hur många fick AI:n rätt?  
Det kallas **Noggrannhet (Accuracy)**.

> **Noggrannhet (Accuracy)** = Antal rätt / Totalt antal frågor  
> *Exempel: 28/30 = 0.93 = 93%*

---
## ⚙️ Del 2 – Förbered data och träna modellen

Vi upprepar stegen från de tidigare lektionerna:

In [ ]:
!pip install xgboost -q
print("✅ xgboost installerat!")

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

# Ladda och dela upp data
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Skapa och träna modellen
modell = XGBClassifier(
    n_estimators=10,
    max_depth=3,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)
modell.fit(X_train, y_train)

print("✅ Modellen är tränad och redo för provet!")

---
## 🔮 Del 3 – Förutsägelser (Predictions)

Nu låter vi AI:n gissa vilken art varje blomma i testdatan är.  
Vi använder metoden **`.predict()`** – precis som i Lektion 3,  
men nu gissar den på **alla 30 testblommor** på en gång!

In [ ]:
# Låt AI:n gissa på testdatan
y_pred = modell.predict(X_test)

# Visa de 10 första gissningarna bredvid de rätta svaren
artnamn = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}

print("AI:ns gissning  |  Rätt svar")
print("-" * 35)
for gissning, ratt in zip(y_pred[:10], y_test[:10]):
    markering = "✓" if gissning == ratt else "✗"
    print(f"{artnamn[gissning]:<16} | {artnamn[ratt]}  {markering}")

---
## 🎯 Del 4 – Noggrannhet (Accuracy)

Hur bra gick det sammanlagt? Vi beräknar **Noggrannheten (Accuracy)**:  
andelen blommor som AI:n gissade rätt.

> **Noggrannhet (Accuracy)** = Andelen korrekta svar  
> 1.0 = 100% = perfekt, 0.5 = 50% = gissar rätt varannan gång

In [ ]:
# Beräkna noggrannheten
noggrannhet = accuracy_score(y_test, y_pred)

print(f"Antal rätt: {int(noggrannhet * len(y_test))} av {len(y_test)}")
print(f"Noggrannhet (Accuracy): {noggrannhet:.1%}")

---
## 🔲 Del 5 – Förväxlingsmatris (Confusion Matrix)

Noggrannheten ger oss ett enda tal – men den berättar inte **vilka** blommor AI:n blandade ihop!

Tänk om AI:n alltid blandar ihop Versicolor och Virginica – men aldrig Setosa?  
Det kan vara viktigt att veta!

Lösningen heter **Förväxlingsmatris (Confusion Matrix)** – ett rutnät som visar:
- Diagonalen (uppifrån vänster till ned höger) = **rätta svar** ✓
- Övriga rutor = **fel** ✗ (AI:n förväxlade en art med en annan)

```
                  AI gissade:
               Setosa  Versic.  Virgin.
Rätt    Setosa  [  10  |   0   |   0  ]
svar:  Versic.  [   0  |   9   |   1  ]
       Virgin.  [   0  |   0   |  10  ]
```

Här ser vi att AI:n förväxlade 1 Versicolor med Virginica – men allt annat stämde!

Nu ritar vi den på riktigt:

In [ ]:
# Rita förväxlingsmatrisen
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=iris.target_names,
    colorbar=False,
    ax=ax
)

ax.set_title("Förväxlingsmatris (Confusion Matrix)\nTestdata", fontsize=13)
plt.tight_layout()
plt.show()

### Hur läser man matrisen?

- **Rader** = de **rätta svaren** (vad blomman faktiskt är)
- **Kolumner** = vad **AI:n gissade**
- **Diagonalen** (de blå rutorna) = AI:n gissade rätt!
- **Utanför diagonalen** = AI:n gissade fel (en förväxling)

#### Exempel: Om en ruta visar `2` i raden "versicolor" och kolumnen "virginica"  
→ Det betyder att 2 Versicolor-blommor felidentifierades som Virginica

#### Varför blandar AI:n ihop just Versicolor och Virginica?

De två arterna ser faktiskt ganska lika ut!  
De har överlappande mätningar – det är svårt att skilja dem åt, även för botanister.

---
## 💬 Del 6 – Reflektionsfrågor och uppgifter

### Frågor att fundera på:

**Fråga 1:** Titta på matrisen.  
Vilken art hade AI:n lättast för att känna igen? Hur kan du se det i matrisen?

**Fråga 2:** Om alla rutor **utanför** diagonalen är nollor – vad betyder det?

**Fråga 3:** Noggrannhet på 100% låter fantastiskt!  
Är 100% noggrannhet alltid möjligt i verkliga problem? Vad tror du?

---

### 🎯 Uppgift: Förändra modellen och se skillnaden

Gå tillbaka till **Del 2** (setup-cellen).  
Ändra `n_estimators=10` till `n_estimators=1` (bara ETT beslutsträd!)  
och ändra `max_depth=3` till `max_depth=1` (ett mycket grunt träd).

Kör sedan **alla celler igen** (Runtime → Run all).

- Vad händer med noggrannheten?
- Vad händer med förväxlingsmatrisen?
- Reflektera: Är fler/djupare träd alltid bättre?

---
## 💡 Del 7 – Sammanfattning

| Begrepp | Förklaring | Engelskt namn |
|---------|------------|---------------|
| **Förutsägelse** | Modellens gissning på ny data | Prediction |
| **Noggrannhet** | Andel rätta svar (0–100%) | Accuracy |
| **Förväxlingsmatris** | Rutnät som visar vilka fel som gjordes | Confusion Matrix |
| **Sann positiv** | AI:n gissade rätt kategori | True Positive |
| **Falsk positiv** | AI:n gissade fel (trodde det var X, var Y) | False Positive |

### Resan hittills:

```
✅ Steg 1: Förstå data     (Lektion 1)
✅ Steg 2: Dela upp data   (Lektion 2)
✅ Steg 3: Träna modellen  (Lektion 3)
✅ Steg 4: Utvärdera       (Lektion 4 – detta!)
⏳ Steg 5: Verklig data    (Lektion 5)
```

### 🚀 Nästa lektion

I **Lektion 5** lämnar vi blommorna bakom oss och går in i verkliga livet –  
vi ska bygga ett AI-system för att **upptäcka kreditkortsbedrägerier**!  
Och vi stöter på ett knepigt problem: vad händer när datan är extremt **obalanserad**?